# CAD Cardiac MRI Dataset — Exploratory Data Analysis

**Dataset:** CAD Cardiac MRI Dataset — Danial Sharifrazi  
**Task:** Binary classification — `Normal` vs `Sick` (Coronary Artery Disease)  
**Modality:** Cardiac MRI (short-axis & long-axis slices, JPEG format)

---

### Contents
1. Setup & Configuration
2. Dataset Scan
3. Class & Patient Distribution
4. Per-Patient Slice Counts
5. Series Structure Analysis
6. Sample Image Visualisation
7. Normal vs Sick Comparison
8. Image Dimension Analysis
9. Pixel Intensity Analysis
10. CLAHE Preprocessing Preview
11. Augmentation Preview
12. Data Quality Check
13. Train / Val / Test Split
14. Summary

## 1 · Setup & Configuration

In [ ]:
import os, random, warnings
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from PIL import Image, ImageEnhance
import cv2

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

CLR_NORMAL = '#2a9d8f'
CLR_SICK   = '#e63946'
CLR_BG     = '#f0f4f8'

# Auto-detect Kaggle vs local
KAGGLE_PATH = '/kaggle/input/datasets/danialsharifrazi/cad-cardiac-mri-dataset'
LOCAL_PATH  = r'C:\Users\yycha\Downloads\dataset'
DATASET_PATH = KAGGLE_PATH if os.path.exists(KAGGLE_PATH) else LOCAL_PATH
CLASSES      = ['Normal', 'Sick']
IMG_SIZE     = 224

print(f'Environment : {"Kaggle" if "kaggle" in DATASET_PATH else "Local"}')
print(f'Dataset path: {DATASET_PATH}')
print(f'Path exists : {os.path.exists(DATASET_PATH)}')

## 2 · Dataset Scan

In [ ]:
def scan_dataset(root):
    """Walk dataset and collect all image paths, patient ids, and series names."""
    images      = defaultdict(list)   # cls -> [path, ...]
    by_patient  = defaultdict(lambda: defaultdict(list))  # cls -> patient -> [path]
    by_series   = defaultdict(lambda: defaultdict(list))  # cls -> series -> [path]

    for cls in CLASSES:
        cls_dir = os.path.join(root, cls)
        for patient in sorted(os.listdir(cls_dir)):
            patient_dir = os.path.join(cls_dir, patient)
            if not os.path.isdir(patient_dir):
                continue
            for dirpath, _, files in os.walk(patient_dir):
                rel   = os.path.relpath(dirpath, patient_dir)
                parts = rel.split(os.sep)
                series_name = parts[0] if parts[0] != '.' else '(root)'
                for f in files:
                    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                        path = os.path.join(dirpath, f)
                        images[cls].append(path)
                        by_patient[cls][patient].append(path)
                        by_series[cls][series_name].append(path)
    return images, by_patient, by_series

print('Scanning dataset…')
images, by_patient, by_series = scan_dataset(DATASET_PATH)

n_normal = len(images['Normal'])
n_sick   = len(images['Sick'])
n_total  = n_normal + n_sick

print(f'\n{"─"*44}')
print(f'  Normal patients : {len(by_patient["Normal"]):>5}')
print(f'  Sick patients   : {len(by_patient["Sick"]):>5}')
print(f'  Normal images   : {n_normal:>5,}')
print(f'  Sick images     : {n_sick:>5,}')
print(f'  Total images    : {n_total:>5,}')
print(f'  Imbalance ratio : {n_normal/n_sick:.2f}:1 (Normal:Sick)')
print(f'{"─"*44}')

## 3 · Class & Patient Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Class Distribution', fontsize=15, fontweight='bold')

# Bar chart — image counts
ax = axes[0]
bars = ax.bar(CLASSES, [n_normal, n_sick],
              color=[CLR_NORMAL, CLR_SICK], alpha=0.85,
              edgecolor='white', linewidth=1.5, width=0.5)
for bar, count in zip(bars, [n_normal, n_sick]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 250,
            f'{count:,}', ha='center', fontweight='bold', fontsize=12)
ax.set_ylabel('Number of Images')
ax.set_title('Image Count per Class')
ax.set_facecolor(CLR_BG)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x):,}'))

# Pie chart — image split
ax = axes[1]
wedges, texts, autotexts = ax.pie(
    [n_normal, n_sick], labels=CLASSES,
    colors=[CLR_NORMAL, CLR_SICK], autopct='%1.1f%%',
    startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 12})
for at in autotexts:
    at.set_fontweight('bold'); at.set_color('white'); at.set_fontsize(13)
ax.set_title('Image Split')

# Bar chart — patient counts
ax = axes[2]
p_counts = [len(by_patient['Normal']), len(by_patient['Sick'])]
bars = ax.bar(CLASSES, p_counts, color=[CLR_NORMAL, CLR_SICK],
              alpha=0.85, edgecolor='white', linewidth=1.5, width=0.5)
for bar, count in zip(bars, p_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(count), ha='center', fontweight='bold', fontsize=14)
ax.set_ylabel('Number of Patients')
ax.set_title('Patient Count per Class')
ax.set_facecolor(CLR_BG)
ax.set_ylim(0, max(p_counts) + 3)

plt.tight_layout()
os.makedirs('static', exist_ok=True)
plt.savefig('static/eda_class_dist.png', bbox_inches='tight')
plt.show()

## 4 · Per-Patient Slice Counts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Slices per Patient', fontsize=14, fontweight='bold')

for ax, cls, color in zip(axes, CLASSES, [CLR_NORMAL, CLR_SICK]):
    patients = sorted(by_patient[cls].keys(),
                      key=lambda d: int(d.replace('Directory_', '')))
    counts   = [len(by_patient[cls][p]) for p in patients]
    labels   = [p.replace('Directory_', 'P') for p in patients]

    bars = ax.barh(labels, counts, color=color, alpha=0.82,
                   edgecolor='white', linewidth=0.8)
    ax.set_title(f'{cls} — {len(patients)} Patients', color=color, fontweight='bold')
    ax.set_xlabel('Number of Slices')
    ax.set_facecolor(CLR_BG)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x):,}'))

    mean_c = np.mean(counts)
    ax.axvline(mean_c, color='#333', linestyle='--', linewidth=1.2, label=f'Mean: {mean_c:.0f}')
    ax.legend(fontsize=9)

    for bar, count in zip(bars, counts):
        ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
                f'{count:,}', va='center', fontsize=8.5)

plt.tight_layout()
plt.savefig('static/eda_patient_counts.png', bbox_inches='tight')
plt.show()

# Print stats
for cls in CLASSES:
    counts = [len(v) for v in by_patient[cls].values()]
    print(f'{cls:8s}  min={min(counts):,}  max={max(counts):,}  mean={np.mean(counts):.0f}  std={np.std(counts):.0f}')

## 5 · Series Structure Analysis

Normal patients have series named `1-short`, `2-long`, `33`, `series000X-Body`.  
Sick patients have series named `SR_X`.  
Understanding this nesting is critical — a naive `os.walk` counts duplicate slices from overlapping subdirs in Normal.

In [ ]:
def get_series_per_patient(cls):
    """Return dict: patient -> list of top-level series dir names."""
    result = {}
    cls_dir = os.path.join(DATASET_PATH, cls)
    for patient in sorted(os.listdir(cls_dir)):
        patient_dir = os.path.join(cls_dir, patient)
        if not os.path.isdir(patient_dir):
            continue
        series = [d for d in os.listdir(patient_dir)
                  if os.path.isdir(os.path.join(patient_dir, d))]
        result[patient] = series
    return result

normal_series = get_series_per_patient('Normal')
sick_series   = get_series_per_patient('Sick')

print('Normal — series per patient:')
for p, s in normal_series.items():
    print(f'  {p}: {len(s)} series  →  {s[:4]}{"..." if len(s) > 4 else ""}')

print('\nSick — series per patient:')
for p, s in sick_series.items():
    print(f'  {p}: {len(s)} series')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Series Count per Patient', fontsize=14, fontweight='bold')

for ax, cls, color, series_dict in zip(
    axes, CLASSES, [CLR_NORMAL, CLR_SICK],
    [normal_series, sick_series]
):
    patients = sorted(series_dict.keys(), key=lambda d: int(d.replace('Directory_', '')))
    counts   = [len(series_dict[p]) for p in patients]
    labels   = [p.replace('Directory_', 'P') for p in patients]

    ax.bar(labels, counts, color=color, alpha=0.82, edgecolor='white')
    ax.set_title(f'{cls}', color=color, fontweight='bold')
    ax.set_xlabel('Patient')
    ax.set_ylabel('Number of Series')
    ax.set_facecolor(CLR_BG)
    for i, (label, count) in enumerate(zip(labels, counts)):
        ax.text(i, count + 0.3, str(count), ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('static/eda_series_counts.png', bbox_inches='tight')
plt.show()

## 6 · Sample Image Visualisation

In [ ]:
def load_gray(path):
    return np.array(Image.open(path).convert('L'))

N_COLS = 8
samples_normal = random.sample(images['Normal'], N_COLS)
samples_sick   = random.sample(images['Sick'],   N_COLS)

fig = plt.figure(figsize=(18, 5))
fig.patch.set_facecolor('#111827')
fig.suptitle('Sample Cardiac MRI Slices', color='white',
             fontsize=15, fontweight='bold', y=1.01)

for row, (samples, cls, color) in enumerate([
    (samples_normal, 'Normal', CLR_NORMAL),
    (samples_sick,   'CAD (Sick)', CLR_SICK)
]):
    for col, path in enumerate(samples):
        ax = fig.add_subplot(2, N_COLS, row * N_COLS + col + 1)
        ax.imshow(load_gray(path), cmap='gray', vmin=0, vmax=255)
        ax.axis('off')
        ax.set_facecolor('black')
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(2)
            spine.set_visible(True)
        if col == 0:
            ax.set_ylabel(cls, color=color, fontsize=11,
                          fontweight='bold', labelpad=6)
            ax.yaxis.set_label_position('left')

plt.tight_layout(pad=0.4)
plt.savefig('static/eda_samples.png', bbox_inches='tight', facecolor='#111827')
plt.show()

## 7 · Normal vs Sick Side-by-Side

In [ ]:
N_PAIRS = 5
pairs_n = random.sample(images['Normal'], N_PAIRS)
pairs_s = random.sample(images['Sick'],   N_PAIRS)

fig, axes = plt.subplots(2, N_PAIRS, figsize=(16, 7))
fig.suptitle('Normal vs CAD — Direct Comparison', fontsize=14, fontweight='bold')

for col, (pn, ps) in enumerate(zip(pairs_n, pairs_s)):
    for row, (path, cls, color) in enumerate([
        (pn, 'Normal', CLR_NORMAL),
        (ps, 'CAD',    CLR_SICK)
    ]):
        ax = axes[row][col]
        ax.imshow(load_gray(path), cmap='gray')
        ax.axis('off')
        ax.set_facecolor('black')
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(2.5)
            spine.set_visible(True)
        if col == 0:
            ax.set_ylabel(cls, color=color, fontweight='bold',
                          fontsize=12, labelpad=6)
        ax.set_title(f'Pair {col+1}', fontsize=9, color='#555')

plt.tight_layout()
plt.savefig('static/eda_comparison.png', bbox_inches='tight')
plt.show()

## 8 · Image Dimension Analysis

In [ ]:
SAMPLE_N = 300
widths  = defaultdict(list)
heights = defaultdict(list)

for cls in CLASSES:
    for path in random.sample(images[cls], min(SAMPLE_N, len(images[cls]))):
        try:
            w, h = Image.open(path).size
            widths[cls].append(w)
            heights[cls].append(h)
        except:
            pass

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Image Dimension Properties', fontsize=14, fontweight='bold')

for cls, color in zip(CLASSES, [CLR_NORMAL, CLR_SICK]):
    aspects = [w/h for w, h in zip(widths[cls], heights[cls])]
    axes[0].hist(widths[cls],  bins=30, alpha=0.65, color=color, label=cls, edgecolor='white')
    axes[1].hist(heights[cls], bins=30, alpha=0.65, color=color, label=cls, edgecolor='white')
    axes[2].hist(aspects,      bins=30, alpha=0.65, color=color, label=cls, edgecolor='white')

axes[0].set_title('Width');        axes[0].set_xlabel('Pixels')
axes[1].set_title('Height');       axes[1].set_xlabel('Pixels')
axes[2].set_title('Aspect Ratio'); axes[2].set_xlabel('W / H')
for ax in axes:
    ax.legend(); ax.set_ylabel('Count'); ax.set_facecolor(CLR_BG)

plt.tight_layout()
plt.savefig('static/eda_dimensions.png', bbox_inches='tight')
plt.show()

print(f'\n{"Class":8s}  {"Width":>12}  {"Height":>12}  {"Aspect":>10}')
print('─' * 48)
for cls in CLASSES:
    aspects = [w/h for w, h in zip(widths[cls], heights[cls])]
    print(f'{cls:8s}  {np.mean(widths[cls]):>6.0f}±{np.std(widths[cls]):<4.0f}  '
          f'{np.mean(heights[cls]):>6.0f}±{np.std(heights[cls]):<4.0f}  '
          f'{np.mean(aspects):>10.3f}')

## 9 · Pixel Intensity Analysis

In [ ]:
SAMPLE_PIX = 200
intensities = defaultdict(list)
std_devs    = defaultdict(list)
pix_flat    = defaultdict(list)

for cls in CLASSES:
    for path in random.sample(images[cls], min(SAMPLE_PIX, len(images[cls]))):
        try:
            arr = np.array(Image.open(path).convert('L'), dtype=np.float32)
            if arr.max() > 0:
                intensities[cls].append(arr.mean())
                std_devs[cls].append(arr.std())
                pix_flat[cls].extend(arr.flatten().tolist())
        except:
            pass

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Pixel Intensity Analysis', fontsize=14, fontweight='bold')

for cls, color in zip(CLASSES, [CLR_NORMAL, CLR_SICK]):
    axes[0].hist(intensities[cls], bins=40, alpha=0.65, color=color, label=cls, edgecolor='white')
    axes[1].hist(std_devs[cls],    bins=40, alpha=0.65, color=color, label=cls, edgecolor='white')
    axes[2].hist(pix_flat[cls],    bins=128, alpha=0.45, color=color, label=cls,
                 density=True, edgecolor='none')

axes[0].set_title('Mean Pixel Intensity per Image'); axes[0].set_xlabel('Mean (0–255)')
axes[1].set_title('Contrast (Std Dev per Image)');   axes[1].set_xlabel('Std Dev')
axes[2].set_title('Pixel Value Distribution');        axes[2].set_xlabel('Pixel Value')
for ax in axes:
    ax.legend(); ax.set_facecolor(CLR_BG)

plt.tight_layout()
plt.savefig('static/eda_intensity.png', bbox_inches='tight')
plt.show()

print(f'\n{"Class":8s}  {"Mean Intensity":>15}  {"Mean Contrast":>14}')
print('─' * 42)
for cls in CLASSES:
    print(f'{cls:8s}  {np.mean(intensities[cls]):>14.2f}  {np.mean(std_devs[cls]):>14.2f}')

In [ ]:
# Average image per class + difference map
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Average MRI Appearance per Class', fontsize=14, fontweight='bold')

avg_imgs = {}
for cls in CLASSES:
    stack = []
    for path in random.sample(images[cls], min(80, len(images[cls]))):
        try:
            arr = np.array(Image.open(path).convert('L').resize((224, 224)), dtype=np.float32)
            if arr.max() > 0:
                stack.append(arr)
        except:
            pass
    avg_imgs[cls] = np.mean(stack, axis=0)

for ax, cls, color, title in zip(
    axes[:2], CLASSES, [CLR_NORMAL, CLR_SICK],
    ['Average Normal MRI', 'Average CAD MRI']
):
    im = ax.imshow(avg_imgs[cls], cmap='gray')
    ax.set_title(title, color=color, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

diff = avg_imgs['Normal'].astype(float) - avg_imgs['Sick'].astype(float)
im = axes[2].imshow(diff, cmap='RdBu_r', vmin=-40, vmax=40)
axes[2].set_title('Difference  (Normal − CAD)', fontweight='bold')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('static/eda_avg_images.png', bbox_inches='tight')
plt.show()

## 10 · CLAHE Preprocessing Preview

CLAHE (Contrast Limited Adaptive Histogram Equalization) is applied to every image before training.  
It enhances local contrast in cardiac MRI without over-amplifying noise.

In [ ]:
def apply_clahe(gray_arr):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(gray_arr.astype(np.uint8))

N = 4
sample_paths = {
    'Normal': random.sample(images['Normal'], N),
    'Sick':   random.sample(images['Sick'],   N),
}

fig, axes = plt.subplots(N * 2, 3, figsize=(12, N * 5))
fig.suptitle('CLAHE Effect on Cardiac MRI Slices', fontsize=14, fontweight='bold')

col_titles = ['Original', 'CLAHE Enhanced', 'Intensity Histogram']
for col, title in enumerate(col_titles):
    axes[0][col].set_title(title, fontsize=11, fontweight='bold', pad=8)

row = 0
for cls, color in zip(CLASSES, [CLR_NORMAL, CLR_SICK]):
    for path in sample_paths[cls]:
        orig = np.array(Image.open(path).convert('L'))
        enhanced = apply_clahe(orig)

        axes[row][0].imshow(orig, cmap='gray', vmin=0, vmax=255)
        axes[row][0].axis('off')
        axes[row][0].set_ylabel(cls, color=color, fontweight='bold', fontsize=9)

        axes[row][1].imshow(enhanced, cmap='gray', vmin=0, vmax=255)
        axes[row][1].axis('off')

        axes[row][2].hist(orig.flatten(),      bins=64, alpha=0.6, color='#64748b', label='Original', density=True)
        axes[row][2].hist(enhanced.flatten(),  bins=64, alpha=0.6, color=color,     label='CLAHE',    density=True)
        axes[row][2].set_xlabel('Pixel Value'); axes[row][2].set_ylabel('Density')
        axes[row][2].legend(fontsize=8); axes[row][2].set_facecolor(CLR_BG)
        axes[row][2].grid(True, alpha=0.3)
        row += 1

plt.tight_layout()
plt.savefig('static/eda_clahe.png', bbox_inches='tight')
plt.show()

## 11 · Augmentation Preview

In [ ]:
import torchvision.transforms.functional as TF
import torch

def augmentation_grid(path):
    img = Image.open(path).convert('L').resize((224, 224))
    arr = np.array(img, dtype=np.float32) / 255.0
    return {
        'Original':      arr,
        'H-Flip':        arr[:, ::-1].copy(),
        'V-Flip':        arr[::-1, :].copy(),
        'Rotate +10°':   np.array(img.rotate(10),  dtype=np.float32) / 255.0,
        'Rotate -10°':   np.array(img.rotate(-10), dtype=np.float32) / 255.0,
        'Brightness ↑':  np.array(ImageEnhance.Brightness(img).enhance(1.3), dtype=np.float32) / 255.0,
    }

fig, axes = plt.subplots(2, 6, figsize=(18, 7))
fig.suptitle('Data Augmentation Preview (applied during training)', fontsize=13, fontweight='bold')

for row, cls in enumerate(CLASSES):
    path = random.choice(images[cls])
    augs = augmentation_grid(path)
    color = CLR_NORMAL if cls == 'Normal' else CLR_SICK
    for col, (aug_name, aug_img) in enumerate(augs.items()):
        ax = axes[row][col]
        ax.imshow(aug_img, cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
        ax.set_title(aug_name, fontsize=9, fontweight='bold')
        for spine in ax.spines.values():
            spine.set_edgecolor(color); spine.set_linewidth(2); spine.set_visible(True)
        if col == 0:
            ax.set_ylabel(cls, color=color, fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('static/eda_augmentation.png', bbox_inches='tight')
plt.show()

## 12 · Data Quality Check

In [ ]:
print('Running quality checks (sampling 500 images per class)…')
QUALITY_N = 500
quality   = defaultdict(lambda: defaultdict(int))
black_examples = []

for cls in CLASSES:
    for path in random.sample(images[cls], min(QUALITY_N, len(images[cls]))):
        try:
            img = Image.open(path).convert('L')
            arr = np.array(img)
            w, h = img.size
            if arr.max() == 0:
                quality[cls]['black'] += 1
                if len(black_examples) < 4:
                    black_examples.append((path, cls))
            elif w < 32 or h < 32:
                quality[cls]['tiny'] += 1
            elif arr.mean() < 3:
                quality[cls]['near_black'] += 1
            else:
                quality[cls]['ok'] += 1
        except Exception:
            quality[cls]['corrupt'] += 1

print(f'\n{"Issue":14}  {"Normal":>8}  {"Sick":>8}')
print('─' * 36)
for issue in ['ok', 'black', 'near_black', 'tiny', 'corrupt']:
    print(f'{issue:14}  {quality["Normal"][issue]:>8}  {quality["Sick"][issue]:>8}')

# Extrapolate to full dataset
print(f'\nEstimated black/near-black in full dataset:')
for cls in CLASSES:
    bad = quality[cls]['black'] + quality[cls]['near_black']
    n   = len(images[cls])
    est = int(bad / QUALITY_N * n)
    print(f'  {cls}: ~{est:,} / {n:,} ({est/n*100:.1f}%)')

In [ ]:
# Quality breakdown bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Data Quality by Class', fontsize=13, fontweight='bold')
issues      = ['ok', 'black', 'near_black', 'tiny', 'corrupt']
issue_colors = ['#2a9d8f', '#e63946', '#f4a261', '#457b9d', '#6d6875']

for ax, cls, color in zip(axes, CLASSES, [CLR_NORMAL, CLR_SICK]):
    vals   = [quality[cls][i] for i in issues]
    bars   = ax.bar(issues, vals, color=issue_colors, alpha=0.85, edgecolor='white')
    ax.set_title(cls, color=color, fontweight='bold')
    ax.set_ylabel('Count (in sample)')
    ax.set_facecolor(CLR_BG)
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    str(v), ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('static/eda_quality.png', bbox_inches='tight')
plt.show()

## 13 · Train / Val / Test Split

Split is **patient-level** (not slice-level) to prevent data leakage.  
A slice-level random split allows the same patient's slices into both train and test, artificially inflating metrics.

In [ ]:
import random as rnd
rnd.seed(42)

def patient_split(cls, train_r=0.75, val_r=0.125):
    patients = sorted(by_patient[cls].keys(),
                      key=lambda d: int(d.replace('Directory_', '')))
    rnd.shuffle(patients)
    n = len(patients)
    n_train = round(n * train_r)
    n_val   = round(n * val_r)
    return (patients[:n_train],
            patients[n_train:n_train + n_val],
            patients[n_train + n_val:])

splits = {}
for cls in CLASSES:
    tr, va, te = patient_split(cls)
    splits[cls] = {'train': tr, 'val': va, 'test': te}

# Compute slice counts per split
split_slices = defaultdict(lambda: defaultdict(int))
for cls in CLASSES:
    for split, patients in splits[cls].items():
        split_slices[split][cls] = sum(len(by_patient[cls][p]) for p in patients)

print(f'{"Split":6}  {"Normal patients":>16}  {"Sick patients":>14}  {"Normal slices":>14}  {"Sick slices":>12}')
print('─' * 72)
for split in ['train', 'val', 'test']:
    np_ = len(splits['Normal'][split])
    sp_ = len(splits['Sick'][split])
    ns_ = split_slices[split]['Normal']
    ss_ = split_slices[split]['Sick']
    print(f'{split:6}  {np_:>16}  {sp_:>14}  {ns_:>14,}  {ss_:>12,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Patient-Level Train / Val / Test Split', fontsize=14, fontweight='bold')

split_colors = {'train': '#457b9d', 'val': '#f4a261', 'test': '#e63946'}

for ax, cls, color in zip(axes, CLASSES, [CLR_NORMAL, CLR_SICK]):
    patients = sorted(by_patient[cls].keys(),
                      key=lambda d: int(d.replace('Directory_', '')))
    labels   = [p.replace('Directory_', 'P') for p in patients]
    counts   = [len(by_patient[cls][p]) for p in patients]

    bar_colors = []
    for p in patients:
        for split, plist in splits[cls].items():
            if p in plist:
                bar_colors.append(split_colors[split])
                break

    ax.barh(labels, counts, color=bar_colors, alpha=0.85, edgecolor='white')
    ax.set_title(f'{cls}', color=color, fontweight='bold')
    ax.set_xlabel('Slices')
    ax.set_facecolor(CLR_BG)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x):,}'))

    patches = [mpatches.Patch(color=c, label=s) for s, c in split_colors.items()]
    ax.legend(handles=patches, fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('static/eda_split.png', bbox_inches='tight')
plt.show()

## 14 · Summary

In [ ]:
total_train = sum(split_slices['train'].values())
total_val   = sum(split_slices['val'].values())
total_test  = sum(split_slices['test'].values())

summary = {
    'Total patients':        f'{len(by_patient["Normal"]) + len(by_patient["Sick"])} (16 Normal, 14 Sick)',
    'Total images':          f'{n_total:,}',
    'Normal images':         f'{n_normal:,} ({n_normal/n_total*100:.1f}%)',
    'Sick (CAD) images':     f'{n_sick:,} ({n_sick/n_total*100:.1f}%)',
    'Imbalance ratio':       f'{n_normal/n_sick:.2f}:1  (handled via class-weighted loss)',
    'Image size':            '240×240 px (original) → resized to 224×224 for model',
    'Format':                'JPEG grayscale',
    'Series naming':         'Normal: series000X-Body / 1-short / 2-long | Sick: SR_X',
    'Preprocessing':         'Grayscale → CLAHE → Resize 224 → 3-channel → ImageNet normalise',
    'Augmentations':         'H-flip, V-flip, Rotation ±10°, Brightness/Contrast ±10%',
    'Split strategy':        'Patient-level (prevents data leakage)',
    'Train slices':          f'{total_train:,}',
    'Val slices':            f'{total_val:,}',
    'Test slices':           f'{total_test:,}',
    'Class weighting':       'Inverse frequency — applied in BCE loss',
    'Notable outlier':       'Directory_28 (Sick) has only 167 slices vs avg ~1,850',
}

print('=' * 60)
print('  CAD CARDIAC MRI — EDA SUMMARY')
print('=' * 60)
for k, v in summary.items():
    print(f'  {k:<22}  {v}')
print('=' * 60)
print(f'\nAll EDA plots saved to static/')

In [ ]:
# Visual summary card
fig = plt.figure(figsize=(16, 5))
fig.patch.set_facecolor('#0f172a')
ax = fig.add_subplot(111)
ax.set_facecolor('#0f172a')
ax.axis('off')

fig.text(0.5, 0.93, 'CAD Cardiac MRI — EDA Summary',
         ha='center', fontsize=17, fontweight='bold', color='white')

stats = [
    ('Total Images', f'{n_total:,}', '#38bdf8'),
    ('Normal', f'{n_normal:,}\n({n_normal/n_total*100:.1f}%)', CLR_NORMAL),
    ('CAD (Sick)', f'{n_sick:,}\n({n_sick/n_total*100:.1f}%)', CLR_SICK),
    ('Patients', '30\n(16N / 14S)', '#a78bfa'),
    ('Image Size', '240×240\n→ 224×224', '#fb923c'),
    ('Imbalance', f'{n_normal/n_sick:.2f}:1\nNormal:Sick', '#facc15'),
    ('Train/Val/Test', f'{total_train:,}\n{total_val:,} / {total_test:,}', '#34d399'),
]

for i, (label, value, color) in enumerate(stats):
    x = 0.07 + i * 0.135
    rect = mpatches.FancyBboxPatch((x - 0.055, 0.08), 0.115, 0.74,
                                    boxstyle='round,pad=0.02', linewidth=0,
                                    facecolor='#1e293b',
                                    transform=fig.transFigure, clip_on=False)
    fig.add_artist(rect)
    fig.text(x, 0.62, value, ha='center', fontsize=12, fontweight='bold',
             color=color, transform=fig.transFigure, va='top')
    fig.text(x, 0.16, label, ha='center', fontsize=9,
             color='#94a3b8', transform=fig.transFigure)

plt.savefig('static/eda_summary.png', bbox_inches='tight', facecolor='#0f172a')
plt.show()